# IFCE Campus Tauá - Tecnologia em Análise e Desenvolvimento de Sistemas (ADS)
## Banco de Dados Não-Relacionais (ADS16) — Aula 05: Padrão Cache-Aside com Redis e TTL
**Prof. Me. Reginaldo Fernandes**  
*Baseado no estudo de caso prático ShopGrids de Renato Augusto ([Assistir no YouTube](https://www.youtube.com/watch?v=vRO0UfvsbDw))*

---
### Roteiro Prático Deste Caderno:
1. **Conexão com o Servidor Redis:** Validação de latência em memória RAM.
2. **Etapa 1 — Sem Cache (O Gargalo no Banco SQL):** Simulando consultas repetidas na tabela `categories WHERE is_featured = 1`.
3. **Etapa 2 — Cache-Aside Básico (`GET` e `SET`):** Aliviando o banco de dados com cópia na memória RAM.
4. **Etapa 3 — O Problema da Inconsistência:** O que acontece quando o banco de dados é atualizado mas o cache não tem expiração?
5. **Etapa 4 — A Solução com TTL (`SETEX`):** Expiração automática e renovação transparente do cache.

### 1. Conexão com o Servidor Redis e Preparação do Banco SQL (`shopgrids.db`)

In [ ]:
import redis
import sqlite3
import json
import time

# 1. Conecta ao Redis local (porta 6379)
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)
print("Conexão com Redis ativa:", r.ping())

# 2. Cria a tabela 'categories' simulando o banco relacional da ShopGrids
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
cursor = conn.cursor()
cursor.execute("CREATE TABLE categories (id INT, name TEXT, is_featured INT)")
cursor.executemany("INSERT INTO categories VALUES (?, ?, ?)", [
    (1, "TV & Áudio", 1),
    (2, "Desktop & Laptop", 1),
    (3, "Smartphones", 1),
    (4, "Acessórios Gamer", 1),
    (5, "Moda Inverno", 0),
    (6, "Aquecedores", 0)
])
conn.commit()

def select_featured_categories():
    time.sleep(0.08)  # Simula 80ms de leitura em disco no banco relacional
    cur = conn.cursor()
    cur.execute("SELECT id, name FROM categories WHERE is_featured = 1")
    return [dict(row) for row in cur.fetchall()]

### 2. Etapa 1 e 2 — Implementando o Padrão Cache-Aside (`GET` + `SET`)

In [ ]:
cache_key = "featured_categories"
r.delete(cache_key)  # Garante que o cache comece vazio

def get_featured_sem_ttl():
    inicio = time.time()
    
    # 1. Consulta o Redis primeiro
    cached = r.get(cache_key)
    if cached:
        tempo = round((time.time() - inicio) * 1000, 2)
        return {"origem": "CACHE_REDIS", "tempo_ms": tempo, "dados": json.loads(cached)}
    
    # 2. Se não encontrou no Redis, busca no Banco SQL e salva no Redis
    dados = select_featured_categories()
    r.set(cache_key, json.dumps(dados))
    tempo = round((time.time() - inicio) * 1000, 2)
    return {"origem": "BANCO_SQL", "tempo_ms": tempo, "dados": dados}

print("1ª Requisição (Cache Miss):", get_featured_sem_ttl())
print("2ª Requisição (Cache Hit): ", get_featured_sem_ttl())

### 3. Etapa 3 — O Problema da Inconsistência (Atualizando o Banco sem TTL no Cache)

In [ ]:
# Chegou o inverno! O time de Produto deixa apenas 'Moda Inverno' e 'Aquecedores' como destaque no banco:
conn.execute("UPDATE categories SET is_featured = 0 WHERE id IN (1, 2, 3, 4)")
conn.execute("UPDATE categories SET is_featured = 1 WHERE id IN (5, 6)")
conn.commit()

print("No Banco SQL agora existem apenas 2 categorias em destaque.")
print("Porém, veja o que a API retorna por causa do SET sem expiração:")
print(get_featured_sem_ttl())

### 4. Etapa 4 — A Solução com Invalidação por Tempo (`SETEX` com TTL)

In [ ]:
# 1. Apagamos a chave travada sem TTL
r.delete(cache_key)

def get_featured_com_ttl(ttl_segundos=10):
    inicio = time.time()
    cached = r.get(cache_key)
    if cached:
        tempo = round((time.time() - inicio) * 1000, 2)
        return {"origem": "CACHE_REDIS", "tempo_ms": tempo, "ttl": r.ttl(cache_key), "dados": json.loads(cached)}
    
    dados = select_featured_categories()
    # SETEX grava a chave já com tempo de vida útil (TTL)!
    r.setex(cache_key, ttl_segundos, json.dumps(dados))
    tempo = round((time.time() - inicio) * 1000, 2)
    return {"origem": "BANCO_SQL", "tempo_ms": tempo, "ttl": ttl_segundos, "dados": dados}

print("1ª Chamada (Busca as 2 categorias de inverno do Banco e grava com TTL de 10s):")
print(get_featured_com_ttl(10))

print("\n2ª Chamada imediata (Vem do Redis super rápido):")
print(get_featured_com_ttl(10))